# Class 13: Visualization continued, and maps

In class 10 we got as far as reviewing the plots we already knew. This class picks up from there and then moves on to maps.

1. **A plot refresher** — the plots we have made so far, and which question each one answers.
2. **Scatter plots with `plt.scatter`** — coloring and sizing points.
3. **Subplots** — several plots in one figure, in both of matplotlib's two interfaces.
4. **matplotlib as a canvas** — drawing things that are not plots.
5. **Seaborn** — a library built on matplotlib that makes good-looking statistical graphics much easier.
6. **Maps** — geopandas, coordinate reference systems, projections, and choropleth maps.

Parts 1 to 5 are the material homework 5 needs. Maps are not on homework 5.

In [ ]:
import YData

# YData.download.download_class_code(13)   # get class code
# YData.download.download_class_code(13, True)  # get the code with the answers

# Data for the visualization part
YData.download_data("movies.csv")

# Data for the maps part
YData.download_data("dennys.csv")
YData.download_data("States_shapefile.geojson")
YData.download_data("state_demographics.csv")
YData.download_data("ne_110m_graticules_10.prj")
YData.download_data("ne_110m_graticules_10.shp")
YData.download_data("ne_110m_graticules_10.shx")
YData.download_data("ne_110m_graticules_10.dbf")

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

## Part 0. Where we were: a plot refresher

Every plot we have made answers a question about a particular *kind* of data. That is the thing worth remembering, more than the function names:

| What you have | What to draw | matplotlib |
|---|---|---|
| one quantitative variable | histogram, box plot | `plt.hist`, `plt.boxplot` |
| one quantitative variable, measured over time | line plot | `plt.plot(x, y)` |
| two quantitative variables | scatter plot | `plt.plot(x, y, ".")`, `plt.scatter(x, y)` |
| one categorical variable | bar chart, pie chart | `plt.bar`, `plt.pie` |
| a quantitative variable split by a categorical one | side-by-side box plots, bar chart of group means | `plt.boxplot([...])`, `plt.bar` |

And whatever you draw, label it: `plt.xlabel`, `plt.ylabel`, `plt.title`, `plt.legend`.

Let's rebuild the Bechdel data from class 10 so we have something to plot.

In [ ]:
# The Bechdel movie data, as in class 10
movies = pd.read_csv("movies.csv")

col_names_to_keep = ['year', 'imdb', 'title', 'clean_test', 'binary', 'budget',
       'domgross', 'budget_2013', 'domgross_2013', 'decade_code', 'imdb_id',
       'rated', 'imdb_rating', 'runtime',  'imdb_votes']

movies = movies[col_names_to_keep]

movies.head()

### Refresher exercise 1: statistics by group

Re-create the `bechdel_stats` DataFrame from class 10: one row for movies that failed the Bechdel test and one for movies that passed, with columns

- `n`: the number of movies
- `mean_revenue`: mean `domgross_2013`
- `mean_budget`: mean `budget_2013`
- `prop`: the proportion of all movies in that row's group

The first three come from one `groupby().agg()`. `prop` is then a column you compute from `n`.

In [ ]:
bechdel_stats = ...

bechdel_stats

### Refresher exercise 2: one quantitative variable, split by a categorical one

Compare the budgets (in 2013 dollars) of movies that passed and failed the Bechdel test, first with side-by-side box plots and then with overlapping histograms.

For the histograms, two arguments earn their place:

1. `alpha = .5` makes them transparent so you can see both.
2. `density = True` scales each histogram to enclose an area of 1. Use it when the two groups have **different numbers of movies**, as they do here — otherwise the taller histogram is just the bigger group.

Which of the two do you think contrasts the budgets more clearly?

In [ ]:
# Split the movies into two DataFrames
bechdel_passed_df = ...
bechdel_failed_df = ...

# Side-by-side box plots
...
plt.ylabel("Budget ($)");

In [ ]:
# Overlapping histograms of the same two groups
...
plt.xlabel("Budget ($)");
plt.ylabel("Density");

### Refresher exercise 3: a time series

Create a DataFrame called `year_revenue` with one row per release `year` and the columns

1. `mean_revenue_2013`: mean revenue in 2013 dollars (`domgross_2013`)
2. `mean_revenue`: mean revenue in the dollars of the day (`domgross`)
3. `n`: the number of movies that year with a recorded revenue

Then plot both revenue columns against the year on one pair of axes, with a legend.

Question to think about: why is the line so choppy before about 1990? Look at `n`.

In [ ]:
# Create the year_revenue DataFrame
...

# Plot the two time series
...
plt.xlabel("Year");
plt.ylabel("Revenue ($)");

## Part 1. Scatter plots

For a quick scatter plot of two quantitative variables we have been using `plt.plot` with a marker and no line:

```python
plt.plot(x, y, ".");
```

When you want the points themselves to carry more information, use `plt.scatter` instead. It takes arguments that `plt.plot` does not:

- `s`: the size of each point
- `color`: the color of each point
- `marker`: the shape of each point
- `alpha`: transparency, between 0 and 1 — useful when points overlap

Let's start with the simple version.

In [ ]:
# Create a basic scatter plot of budget vs. revenue

...
plt.xlabel("Budget");
plt.ylabel("Revenue");

Let's now create a DataFrame called `movies2` which is the same as our `movies` DataFrame but it also has an additional column called `before2000`. We will use this column to plot points before the year 2000 in a different color than points after 2000. 

In particular, the `before2000` column should be set to the string "red" for movies that occur before 2000 and to the string "blue" for points that occurred after 2000. 

We can then use the `plt.scatter()` function to plot data colored by whether a movie was made before or after 2000.

Do the results show anything interesting/interpretable? 


Remark: red/blue is more colorblind friendly than red/green.

In [ ]:
# Add a column called "before2000" which has values that are
# "red" for years before 2000, "blue" for years after 2000

movies2 = movies.copy()

...


In [ ]:
...

plt.xlabel("Budget");
plt.ylabel("Revenue");

## Part 2. Subplots

Often you want several plots side by side in one figure. matplotlib gives you two ways to do this, and it is worth seeing both because you will meet both in other people's code.

**1. The pyplot interface.** `plt.subplot` says "the figure is a grid this big; from now on I am drawing in this cell of it". Every `plt.`-something afterwards applies to that cell until you call `plt.subplot` again.

```python
plt.subplot(num_rows, num_cols, curr_plot_num);
plt.plot(x, y);
```

The plots are numbered from 1, reading left to right and then top to bottom.

**2. The axes interface.** `plt.subplots` — note the **s** — makes the whole grid at once and hands you back the figure plus an object for each cell. You then draw on a cell by name.

```python
fig, ax = plt.subplots(num_rows, num_cols);
ax[0].plot(x1, y1);
ax[1].plot(x2, y2);
```

Two things to watch with the axes interface:

- `ax` is an **array** of axes, so you must index it: `ax[0]`, `ax[1]`. `ax.plot(...)` on the array itself is an error.
- The labeling methods are named differently: on an axis it is `ax[0].set_ylabel("...")`, not `plt.ylabel("...")`.

Let's draw the same two plots — a bar chart and a pie chart of `prop` — both ways.

In [ ]:
# reset the index to make "binary" a column in the DataFrame
...

# change the figure size
...

# subplots
...
plt.ylabel("Proportion");

...


Now the same figure through the axes interface. Notice `plt.subplots` with the **s**, the indexing into `ax`, and `set_ylabel` in place of `ylabel`.

In [ ]:
# using the axes interface

...

# set the figure size
...    
# subplots
...


## Part 3. matplotlib as a general canvas

We can use matplotlib as a general canvas to create illustrations as well. For example, in the YData baseball class we drew a baseball diamond and updated where runners were on base.

Let's explore this very briefly...

In [ ]:
        
# draw the baseball diamond

# make the axes square

# put a runner on first

# turn off the axis

# make the field green 


## Part 4. Seaborn

[Seaborn](https://seaborn.pydata.org/index.html) is a Python data visualization library built on top of matplotlib. It produces better-looking plots with less code, and it is built around the idea of handing it a whole DataFrame plus the *names* of the columns you want plotted, rather than handing it arrays.

The first thing it gives you is a theme. `sns.set_theme()` changes the look of **every** plot from then on, including your matplotlib ones.

In [ ]:
# Import seaborn
import seaborn as sns

# Apply the default theme
sns.set_theme()   # default style is 'darkgrid')
#sns.set_theme(style='whitegrid')

# Side note: Matplotlib also has themes
# plt.style.available
# plt.style.use('fivethirtyeight')


# Re-draw the revenue line plot, now with the seaborn theme applied
...

plt.legend();
plt.ylabel("Revenue ($)");

### Figure-level functions

Seaborn has two sorts of plotting function: *axes-level* functions that draw on a single axis, and *figure-level* functions that manage a whole figure for you. We will use the figure-level ones, and there are three, grouped by the kind of data you have:

| What you have | Function |
|---|---|
| two quantitative variables | `sns.relplot()` |
| one quantitative variable | `sns.displot()` |
| a quantitative variable across levels of a categorical one | `sns.catplot()` |

This is the same table as Part 0, with seaborn names in the right-hand column. Each one takes `data = `, then column *names* as strings.

### Two quantitative variables: `sns.relplot()`

Let's plot movie revenue against budget again, this time in seaborn.

In [ ]:
# plot relationship between budget and revenue (domgross_2013)



#### Penguins!

Let's continue to explore the relplot using data on penguins. 

We will also look at mapping other features of our data onto visual properties including: 
- `x`, and `y` column names to be plotted (as we have done before)
- `hue`: The column name to be mapped to the color of the points
- `size`: The column name to be mapped to the size of points
- `style`: The column name to be mapped to the style of the markers
- `col`: the column name to be mapped to faceting to compare multiple subplots

In [ ]:
# Let's look at some penguins
penguins = sns.load_dataset("penguins")

print(type(penguins))

penguins.head()

In [ ]:
# plotting bill size on x, and y axes and other properties



### Plotting a single quantitative variable using sns.displot()

We can plot a single quantitative variables using the `sns.displot()` function.

Properties we can set include
- `x`: The name of the data column you want to plot
- `hue`: The name of the column that gives each group its own color
- `kind` The type of plot

Different options for `kind` are: “hist”, “kde”, “ecdf”


#### Warm-up exercise

Please create a `sns.displot()` to create a visualization of *flipper length*, where each *species* is in a different color (i.e., different hue). Also, experiment with the "kind" of visualization and choose the kind you think creates the best visualization. 

In [ ]:
# plot the flipper length
...


### Plotting a quantitative variable for different categorical variable levels

We can plot a quantitative variable for different categorical variable levels using the `sns.catplot()` function.

We specify: 
- `x`: categorical x-value column name
- `y`: Quantitative y-value column name
- `kind`: The type of plot

The `kind` argument can be set to the following: “strip”, “swarm”, “box”, “violin”, “boxen”, “point”, “bar”, or “count”

In [ ]:
# plot flipper length for the different species using different kinds of plots


# also try “strip”, “swarm”, “box”, “violin”, “boxen”, “point”, or “bar”

### Pairs plots

One of the most useful visualizations for exploring the relationships between several quantitative variables is to create a "pairs plot" which creates a series of scatter plots between all quantitative variables in the data.  We can do this in seaborn using the `sns.pairplot(data)` function!


Use the `pairplot()` function to visualize the relationships between all columns in the `penguins` DataFrame. Also, make each species have a different color. 

In [ ]:
# Create pair plots for the different variables in the penguins data set

...


<img src = "https://i.imgflip.com/1ezfdq.jpg">

### Spatial mapping with geopandas

Visualizing spatial data through maps is another powerful way to see trends in data. There are several mapping packages in Python. Here we will use the geopandas package to create maps. 

The geopandas package defines a geopandas DataFrame, which is the same as a pandas DataFrame but has an additional column called `geometry` which specifies geographic information. 

Let's explore this now!

### Visualizing boundaries

Let's start by looking at some geopandas DataFrames and visualizing some geometric boundaries.

In [ ]:
import geopandas as gpd

# see which maps come with geopandas
gpd.datasets.available
# Oh no! They don't provide datasets anymore!

Let's get a geopandas DataFrame that has the countries in the world...

In [ ]:
# View the world geopandas DataFrame
# read data into a geodataframe
# NOTE: this downloads ~215 KB from naciscdn.org when the cell runs.
# Once ne_110m_admin_0_countries.zip is in the course data repo, replace
# these two lines with:  world = gpd.read_file("ne_110m_admin_0_countries.zip")
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
# In previous versions of geopandas, the dataset was included with the package
#world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# print the data type
print(type(world))

# look at the first few rows of the data
world.head()

In [ ]:
# Plot a world map with particular properties

...


In [ ]:
world.query("ADMIN == 'United States of America'")

In [ ]:
world.query("SOVEREIGNT == 'United States of America'")

In [ ]:
# Plot just the United States

...


### Coordinate reference systems and projections

A coordinate reference system (CRS) is a framework used to precisely measure locations on the surface of the Earth as coordinates. The goal of any spatial reference system is to create a common reference frame in which locations can be measured precisely and consistently as coordinates, which can then be shared unambiguously, so that any recipient can identify the same location that was originally intended by the originator.

There are two different types of coordinate reference systems: Geographic Coordinate Systems and Projected Coordinate Systems. [Projected coordinate systems](https://en.wikipedia.org/wiki/List_of_map_projections) map 3D coordinates into a 2D plane so they can be plotted. Different projected coordinate systems preserve different properties, such as keeping all angles intact which is useful for navigation (e.g., the Mercator projection) or keeping the size of land areas intact (e.g., the Eckert IV projection). 

A detailed discussion of CRS is beyond the scope of the class. But for the purposes of this class, it is just important that all layers in a map are using the same project (otherwise, for example, data points representing cities and the underlying spatial map won't line up). 

Let's very briefly explore different map projections... 

In [ ]:
# Read Graticules (lines on a map)
graticules = gpd.read_file("ne_110m_graticules_10.shp")
print(graticules.crs)
graticules.head(3)

In [ ]:
# Web Mercator projection - preserves angles (EPSG:4326 projection)

print(world.crs) # print the default CRS


# plot the map
...


In [ ]:
# Eckert IV is an equal-area projection  ("ESRI:54012")

...


In [ ]:
# Robinson projection - neither equal-area nor conformal ("ESRI:54030") 

...


To learn more about "What your favorite map projection says about you" see: https://xkcd.com/977/

### Maps with layers and markers

We can also plot points on a map. When doing so, it's important that the points and the underlying map use the same coordinate reference system (CRS).

Let's add Denny's locations to the map of the United States!

In [ ]:
# Let's start by getting a map of just the United States

...


In [ ]:
# visualize just the United States

...


In [ ]:
# Get the coordinate reference system (CRS) for our map

...


Let's now load our Denny's data!

In [ ]:
# Let's load our Denny's data
dennys = pd.read_csv("dennys.csv")
dennys.head(3)

To convert longitude and latitude coordinates into geometric objects; i.e., we will convert them into Shaply objects.  We can use the `gpd.points_from_xy(long, lat)` function. 

In [ ]:
# Let's convert our longitude and latitude coordinates into geometric (Shapely) objects 

...


In [ ]:
# Let's now convert out data into a geopandas DataFrame

...


In [ ]:
# We can plot the location of the Denny's using the plot function

...


In [ ]:
# Let's check the CRS

...


Before plotting data, we should set the appropriate coordinate reference system (CRS). This is particularly important when we are combining different layers on a map, such as putting city locations on the map that has the outlines of regional borders. 

The CRS that uses longitude and latitude coordinates is the [World Geodetic System 1984 (WGS84)](https://epsg.io/4326). This system is often referred to by its EPSG Geodetic Parameter Dataset code which is `4326`. 

Thus, we should set the set coordinate system to be EPSG 4326. We can do this using the method `.set_crs(4326)`. Let's set this on our `dennys_gpd` DataFrame. 

In [ ]:
# Let's set the CRS to match the CRS of our map (which is EPSG 4326)

...


Now that we have our Denny's location in the same coordinate system as our map, we can add the points to the map. 

In [ ]:

...


### Choropleth maps

In choropleth maps, predefined regions are filled in with colors based on values of interest. 

Typically to create a choropleth map we join data of interest onto a map. 

Let's explore this now...

In [ ]:
import plotly.express as px

gapminder_2007 = px.data.gapminder().query("year == 2007")   # We only want plotly for its copy of the gapminder data: unlike our own
# gapminder_data.csv it carries the iso_alpha and iso_num country codes,
# which we need further down to join onto the map reliably.

gapminder_2007.head()

In [ ]:
# Join the gapminder data onto our world map

...


In [ ]:
# Plot a choropleth map of life expectancy



In [ ]:
# Change the color scale



In [ ]:
# We can plot quantiles



#### BUT WAIT! Are we missing some countries??? Let's try to find out why.

New commands we'll try here are not for the exam.

How many countries did each join keep? And does the map change once we join on the numeric code instead of the name?

### Another choropleth map example

Let's fit a choropleth map examining which states in the USA are growing in terms of people having lots of children. 

Any thoughts on which state this might be? 

To start, let's load a map with the outlines of the states in the USA, and load demographic data.

In [ ]:
state_map = gpd.read_file("States_shapefile.geojson")

print(state_map.crs)

state_map.head(3)

In [ ]:
# load demographic data on the states

state_demographics = pd.read_csv("state_demographics.csv")
state_demographics.head(3)

In [ ]:
# In order to join the DataFrames, we need to make sure the states have the same capitalization

...


In [ ]:
# Join the demographic information on to the the USA map

...


In [ ]:
# Let's plot the map 

...


Is there anything [wrong with this map](https://xkcd.com/1138/)? 

In [ ]:
# Let's look at the proportion of people under the age of 5

...


In [ ]:
# Let's plot the new map

...


In [ ]:
# But what does it mean?
...
